In [2]:
import polars as pl

In [91]:
my_szshot = pl.read_parquet("/data/xujiayi/xjy/l2/proc/20260624/szshot_1m.pq")

szshot = pl.read_parquet("/data/xujiayi/xjy/l2/raw/20260624/szshot.pq")
szshot= szshot.filter(pl.col('MDStreamID') == 10).with_columns(pl.col('UpdateTime').str.to_time())
cols = ['UpdateTime', 'SecurityID']
for i in range(1, 11):
    cols.extend([f'BidPrice{i}', f'BidVolume{i}', f'AskPrice{i}', f'AskVolume{i}'])
szshot = szshot.select(cols).sort(['SecurityID','UpdateTime'])

In [92]:
szshot = szshot.with_columns([
    pl.col('UpdateTime').dt.hour().alias('hour'),
    pl.col('UpdateTime').dt.minute().alias('minute'),
]).with_columns(
    pl.col('UpdateTime').max().over(['SecurityID','hour','minute']).alias('max_time')
).filter(
    pl.col('UpdateTime') == pl.col('max_time')
).drop('max_time')

In [97]:
MINUTE_NS = 60 * 1_000_000_000

szshot = szshot.with_columns(
    (
        (
            pl.col("UpdateTime").cast(pl.Int64)
            + MINUTE_NS - 1
        )
        // MINUTE_NS
        * MINUTE_NS
    )
    .cast(pl.Time)
    .alias("BarTime")
    ).select(
        "BarTime",
        pl.all().exclude(["BarTime",'UpdateTime'])
)

In [105]:
szshot.filter((pl.col('BarTime')<=pl.time(15,0))).filter(pl.col('SecurityID')==1).sort('BarTime')[60:]

BarTime,SecurityID,BidPrice1,BidVolume1,AskPrice1,AskVolume1,BidPrice2,BidVolume2,AskPrice2,AskVolume2,BidPrice3,BidVolume3,AskPrice3,AskVolume3,BidPrice4,BidVolume4,AskPrice4,AskVolume4,BidPrice5,BidVolume5,AskPrice5,AskVolume5,BidPrice6,BidVolume6,AskPrice6,AskVolume6,BidPrice7,BidVolume7,AskPrice7,AskVolume7,BidPrice8,BidVolume8,AskPrice8,AskVolume8,BidPrice9,BidVolume9,AskPrice9,AskVolume9,BidPrice10,BidVolume10,AskPrice10,AskVolume10,hour,minute
time,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i8,i8
09:16:00,1,"""10.720000""","""65100""","""10.720000""","""65100""",null,null,"""0.000000""","""1800""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,9,15
09:17:00,1,"""10.720000""","""67300""","""10.720000""","""67300""",null,null,"""0.000000""","""3700""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,9,16
09:18:00,1,"""10.720000""","""83000""","""10.720000""","""83000""","""0.000000""","""12200""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,9,17
09:19:00,1,"""10.720000""","""84500""","""10.720000""","""84500""","""0.000000""","""12600""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,9,18
09:20:00,1,"""10.720000""","""83700""","""10.720000""","""83700""","""0.000000""","""21600""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,9,19
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
14:57:00,1,"""10.530000""","""100""","""10.540000""","""99000""","""10.520000""","""108800""","""10.550000""","""196300""","""10.510000""","""1041400""","""10.560000""","""60100""","""10.500000""","""1593400""","""10.570000""","""153100""","""10.490000""","""138400""","""10.580000""","""131700""","""10.480000""","""405100""","""10.590000""","""111800""","""10.470000""","""119100""","""10.600000""","""119500""","""10.460000""","""79000""","""10.610000""","""56900""","""10.450000""","""375200""","""10.620000""","""120800""","""10.440000""","""49900""","""10.630000""","""85000""",14,56
14:58:00,1,"""10.540000""","""377823""","""10.540000""","""377823""","""0.000000""","""277""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,14,57
14:59:00,1,"""10.550000""","""535300""","""10.550000""","""535300""",null,null,"""0.000000""","""249623""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,14,58


In [101]:
my_szshot.filter(pl.col('SecurityID') == 1)

BarTime,SecurityID,BidPrice1,BidQty1,AskPrice1,AskQty1,BidPrice2,BidQty2,AskPrice2,AskQty2,BidPrice3,BidQty3,AskPrice3,AskQty3,BidPrice4,BidQty4,AskPrice4,AskQty4,BidPrice5,BidQty5,AskPrice5,AskQty5,BidPrice6,BidQty6,AskPrice6,AskQty6,BidPrice7,BidQty7,AskPrice7,AskQty7,BidPrice8,BidQty8,AskPrice8,AskQty8,BidPrice9,BidQty9,AskPrice9,AskQty9,BidPrice10,BidQty10,AskPrice10,AskQty10
time,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
09:16:00,1,11.78,10300.0,9.64,3300.0,11.23,1000.0,10.02,100.0,10.84,2200.0,10.3,400.0,10.8,800.0,10.41,800.0,10.79,1100.0,10.5,500.0,10.76,7400.0,10.55,100.0,10.75,6000.0,10.62,32200.0,10.73,10500.0,10.64,6000.0,10.72,25700.0,10.65,100.0,10.71,10200.0,10.68,900.0
09:17:00,1,11.78,10500.0,9.64,3600.0,11.23,1000.0,10.02,100.0,10.84,2200.0,10.3,400.0,10.8,800.0,10.4,1800.0,10.79,1100.0,10.41,800.0,10.76,7400.0,10.5,500.0,10.75,6000.0,10.55,100.0,10.73,10500.0,10.62,32200.0,10.72,27800.0,10.64,6000.0,10.71,11000.0,10.65,100.0
09:18:00,1,11.78,12300.0,9.64,14600.0,11.57,300.0,10.02,100.0,11.23,1000.0,10.3,400.0,11.0,400.0,10.4,1800.0,10.91,100.0,10.41,800.0,10.84,2200.0,10.5,500.0,10.8,800.0,10.55,100.0,10.79,1100.0,10.62,32200.0,10.76,7400.0,10.64,6000.0,10.75,6000.0,10.65,100.0
09:19:00,1,11.78,12300.0,9.64,14600.0,11.57,300.0,10.02,100.0,11.23,1000.0,10.11,1200.0,11.0,400.0,10.3,400.0,10.91,100.0,10.4,1800.0,10.84,2200.0,10.41,800.0,10.8,800.0,10.5,500.0,10.79,1100.0,10.55,100.0,10.76,7400.0,10.62,32200.0,10.75,6000.0,10.64,6000.0
09:20:00,1,11.78,12300.0,9.64,14600.0,11.57,300.0,10.02,100.0,11.23,1000.0,10.11,1200.0,11.0,400.0,10.3,400.0,10.91,100.0,10.4,1800.0,10.84,2200.0,10.41,800.0,10.8,800.0,10.5,500.0,10.79,1100.0,10.55,100.0,10.76,7400.0,10.62,32200.0,10.75,6700.0,10.64,6000.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
14:56:00,1,10.52,95600.0,0.0,1200.0,10.51,1.0419e6,10.53,29900.0,10.5,1.6335e6,10.54,157700.0,10.49,153900.0,10.55,254200.0,10.48,406600.0,10.56,60100.0,10.47,125500.0,10.57,154300.0,10.46,79300.0,10.58,131700.0,10.45,376100.0,10.59,111900.0,10.44,54100.0,10.6,119500.0,10.43,191800.0,10.61,56900.0
14:57:00,1,10.53,100.0,10.54,39900.0,10.52,111300.0,10.55,196300.0,10.51,1.0414e6,10.56,46100.0,10.5,1.5935e6,10.57,120400.0,10.49,138400.0,10.58,131700.0,10.48,405100.0,10.59,111800.0,10.47,119100.0,10.6,119500.0,10.46,79000.0,10.61,56900.0,10.45,375200.0,10.62,120800.0,10.44,49900.0,10.63,85000.0
14:58:00,1,11.78,3500.0,9.64,12000.0,11.06,12400.0,9.65,9600.0,11.0,100.0,10.23,900.0,10.86,25300.0,10.33,184923.0,10.85,15900.0,10.34,85200.0,10.75,266200.0,10.35,9600.0,10.57,3100.0,10.36,16000.0,10.56,5400.0,10.4,5000.0,10.55,10000.0,10.45,1000.0,10.54,49600.0,10.5,7600.0
